# Keeping track of information
LLMs calls are inherently stateless - they do not have memory of any previous interactions. Every call is an independent event, and **YOU** must manage any information that needs to be carried over time.

In this notebook we will look at a few different things that we might want to keep track of between calls.

## Conversation History
Keeping track of the conversation history is actually easy. Firstly it is important to remember that LLMs often have the following pattern:

```
-> system prompt
-> user prompt
-> model response

-> user prompt
-> model response

-> user prompt
-> model response

-> etc.
```

We actually saw an example of this in the prompting notebook when we looked at few-shot prompting.

Here is a really simple example of how we can keep track of the conversation history. We can first define a `system_state` dictionary that will store important information for us. We can give it a `conversation_history` key that will store the conversation history.

In [1]:
system_state = {
    "conversation_history": []
}

In [2]:
system_prompt = (
    "You are a helpful philosophical assistant. "
    "You will help me think about philosophical questions. "
    "Please keep your answers concise and to the point."
)

system_state["conversation_history"].append({
    "role": "system",
    "content": system_prompt
})

user_prompt = "What is the meaning of life?"

system_state["conversation_history"].append({
    "role": "user",
    "content": user_prompt
})

In [3]:
for message in system_state["conversation_history"]:
    print(f"{message['role']}: {message['content']}\n")

system: You are a helpful philosophical assistant. You will help me think about philosophical questions. Please keep your answers concise and to the point.

user: What is the meaning of life?



Now we can use this conversation history to generate a response.

In [4]:
from openai import OpenAI
client = OpenAI()

import dotenv
import os

dotenv.load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

from rich.pretty import pprint

In [5]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=system_state["conversation_history"],
    max_tokens=512,
    temperature=1.0
)

print(response.choices[0].message.content)

The meaning of life is a highly subjective question and can vary greatly among individuals. Some philosophical perspectives suggest that meaning is derived from personal experiences, relationships, or contributions to society, while others propose that it is an inherent aspect of the universe or a pursuit of knowledge and understanding. Ultimately, many believe it is up to each person to create their own meaning based on their values and beliefs.


Great, but now what happens if I want to ask a follow up question? Without the conversation history?

In [6]:
follow_up_prompt = "Can you tell more about point 1?"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": follow_up_prompt
    }],
    max_tokens=512,
    temperature=1.0
)

print(response.choices[0].message.content)


It seems like you're referring to a specific point, but I don't have access to previous messages or context. Could you please clarify what "point 1" refers to? This way, I can provide more detailed information on the topic you're interested in.


Obviously it has no memory of the previous conversation. So we just need to append the follow up prompt to the conversation history.

In [7]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=system_state["conversation_history"],
    max_tokens=512,
    temperature=1.0
)

system_state["conversation_history"].append({
    "role": "assistant",
    "content": response.choices[0].message.content
})

for message in system_state["conversation_history"]:
    print(f"{message['role'].upper()}: {message['content']}\n")

SYSTEM: You are a helpful philosophical assistant. You will help me think about philosophical questions. Please keep your answers concise and to the point.

USER: What is the meaning of life?

ASSISTANT: The meaning of life is a deeply personal and subjective question. Different philosophical perspectives offer various interpretations: 

1. **Existentialism** suggests that individuals create their own meaning through choices and actions.
2. **Utilitarianism** emphasizes maximizing happiness and reducing suffering as a purpose.
3. **Religious views** often provide a divine or spiritual framework for understanding life's significance.
4. **Humanism** focuses on human welfare and the pursuit of knowledge, relationships, and fulfillment.

Ultimately, the meaning of life may be whatever you choose to make of it based on your beliefs, values, and experiences.



And now we can keep the conversation going in a simple loop. If you run this cell a few times you will see that the conversation history is correctly maintained.

In [8]:
while True:
    user_input = input("You: ")
    
    if user_input.lower() in ['exit', 'quit', 'bye']:
        print("Assistant: Goodbye!")
        break
    
    system_state["conversation_history"].append({
        "role": "user",
        "content": user_input
    })
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=system_state["conversation_history"],
        max_tokens=512,
        temperature=1.0
    )
    
    assistant_response = response.choices[0].message.content
    print(f"Assistant: {assistant_response}\n")
    
    system_state["conversation_history"].append({
        "role": "assistant",
        "content": assistant_response
    })

Assistant: Sure! Existentialism is a philosophical movement that emphasizes individual existence, freedom, and choice. Key points include:

1. **Existence Precedes Essence**: This fundamental tenet suggests that individuals are not born with predetermined purposes; rather, they create their own essence through their actions and decisions.

2. **Freedom and Responsibility**: Existentialists argue that humans are fundamentally free and responsible for their choices. This freedom can be both liberating and burdensome, as it places the weight of meaning-making on the individual.

3. **Authenticity**: Living authentically means being true to oneself, rather than conforming to societal expectations. Authenticity is achieved by embracing one's freedom and making choices that reflect personal values.

4. **Absurdity**: Many existentialists, like Albert Camus, discuss the "absurd" — the conflict between the human desire for meaning and the indifferent universe. Rather than despairing, individua

In [9]:
from rich.console import Console
from rich.text import Text

console = Console()

In [10]:
colors = {
    "system": "green",
    "user": "cyan",
    "assistant": "magenta"
}

for message in system_state["conversation_history"]:
    role = message["role"]
    content = message["content"]
    color = colors[role]
    console.print(f"[{color}]{role.upper()}: {content}[/{color}]")

SYSTEM: You are a helpful philosophical assistant. You will help me think about philosophical questions. Please 
keep your answers concise and to the point.

USER: What is the meaning of life?

ASSISTANT: The meaning of life is a deeply personal and subjective question. Different philosophical perspectives 
offer various interpretations: 

1. **Existentialism** suggests that individuals create their own meaning through choices and actions.
2. **Utilitarianism** emphasizes maximizing happiness and reducing suffering as a purpose.
3. **Religious views** often provide a divine or spiritual framework for understanding life's significance.
4. **Humanism** focuses on human welfare and the pursuit of knowledge, relationships, and fulfillment.

Ultimately, the meaning of life may be whatever you choose to make of it based on your beliefs, values, and 
experiences.

USER: Can you tell me more about point 1?

ASSISTANT: Sure! Existentialism is a philosophical movement that emphasizes individual existence, freedom, and 
choice. Key points include:

1. **Existence Precedes Essence**: This fundamental tenet suggests that individuals are not born with predetermined
purposes; rather, they create their own essence through their actions and decisions.

2. **Freedom and Responsibility**: Existentialists argue that humans are fundamentally free and responsible for 
their choices. This freedom can be both liberating and burdensome, as it places the weight of meaning-making on the
individual.

3. **Authenticity**: Living authentically means being true to oneself, rather than conforming to societal 
expectations. Authenticity is achieved by embracing one's freedom and making choices that reflect personal values.

4. **Absurdity**: Many existentialists, like Albert Camus, discuss the "absurd" — the conflict between the human 
desire for meaning and the indifferent universe. Rather than despairing, individuals are encouraged to confront 
this absurdity and create their own meaning.

5. **Notable Thinkers**: Key figures include Jean-Paul Sartre, Simone de Beauvoir, and Franz Kafka, each exploring 
different dimensions of human freedom, choice, and the search for meaning.

In essence, existentialism posits that while life may be inherently meaningless, individuals can forge significance
through their experiences and choices.

## Tracking tokens
We should probably also track the tokens. This can be useful for a few reasons - we can track costs, and we can use it to cut off conversation history when we get too close to our limit.

We can make this as simple or complicated as we want. Probably we should create a `Conversation` class to keep track of things like this.

In [11]:
class Conversation:
    def __init__(self, system_prompt):
        self.system_prompt = system_prompt
        self.history = []
        self.tokens = 0
        self.token_limit = 300

        self.add_message("system", system_prompt)

    def add_message(self, role, content):
        self.history.append({"role": role, "content": content})
        self.tokens += len(content)

    def check_token_limit(self):
        while self.tokens > self.token_limit and len(self.history) > 1:
            # Remove the oldest non-system message
            for i in range(1, len(self.history)):
                if self.history[i]["role"] != "system":
                    removed_message = self.history.pop(i)
                    self.tokens -= len(removed_message["content"])
                    break

    def response(self, user_input):
        self.add_message("user", user_input)
        if self.tokens > self.token_limit:
            self.check_token_limit()
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=self.history,
            max_tokens=512,
            temperature=1.0
        ).choices[0].message.content
        
        self.add_message("assistant", response)

        return response
                

Let's see how this works.

Let's see if we can get the model to forget about things we mention at the start of a conversation.

In [12]:
conversation = Conversation(system_prompt)

In [13]:
print(conversation.response("Hello, my name is Bob and I am 25 years old!"))
print(f"Tokens: {conversation.tokens}")

Hello, Bob! It’s nice to meet you. What philosophical questions or topics are you interested in discussing?
Tokens: 298


In [14]:
print(conversation.response("What is my name?"))
print(f"Tokens: {conversation.tokens}")


You mentioned your name is Bob. Would you like to discuss any philosophical questions related to identity or names?
Tokens: 385


Great, so now we have hit our token limit, and the conversation should be trimmed in the next response.

In [15]:
print(conversation.response("What is my age?"))
print(f"Tokens: {conversation.tokens}")

I don't have information about your age. If you're interested, we could discuss the philosophical implications of age and how it relates to identity or experience.
Tokens: 456


In [16]:
pprint(conversation.history, expand_all=True)

[
│   {
│   │   'role': 'system',
│   │   'content': 'You are a helpful philosophical assistant. You will help me think about philosophical questions. Please keep your answers concise and to the point.'
│   },
│   {
│   │   'role': 'user',
│   │   'content': 'What is my name?'
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': 'You mentioned your name is Bob. Would you like to discuss any philosophical questions related to identity or names?'
│   },
│   {
│   │   'role': 'user',
│   │   'content': 'What is my age?'
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': "I don't have information about your age. If you're interested, we could discuss the philosophical implications of age and how it relates to identity or experience."
│   }
]

This is a good start, but there is a problem here. What if there was something very important that we wanted to keep track of that was mentioned at the start of the conversation, but it has been cut off!?